# Method convergence: measurements, hypotheses and successive orders

Each method tutorial first writes and solves its own local and global equations. This companion lesson recomputes the final qualification plots from the attributed numerical records. It does not run their PDE acquisitions or claim that rendering archived coefficients is a current acquisition.

Select the method and check its approximation spaces, physical observable, refinement variable and source SHA256 before interpreting a slope. A spatial energy theorem, a temporal integration order and a difference to a numerical time reference are different statements.

In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/e86d39320139d73212afc9beb927078fc1f3bb17d31fd716025918e7bc0d7229/method_rates-companion.zip"
COMPANION_SHA256 = "e86d39320139d73212afc9beb927078fc1f3bb17d31fd716025918e7bc0d7229"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("convergence/method_rates.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("convergence/method_rates.ipynb")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.tutorial_convergence import method_series, plot_method_series

studies = method_series(ROOT)
print("Qualified methods:", [study["method"] for study in studies])

## 1. Read the exact refinement variable and physical norm

For consecutive independently measured errors, compute

$$
r_i=\frac{\log(E_{i-1}/E_i)}{\log(h_{i-1}/h_i)}.
$$

The shared reader verifies positive finite errors and at least four distinct decreasing refinement sizes. It reads the actual PyMHM norm records, not values digitized from a paper figure. Its returned metadata retain source attribution and the distinction between literature orders and reported observations.

In [ ]:
method = "unfitted"  # Change this to any name printed above.
study = next(item for item in studies if item["method"] == method)
for key in ("record", "sha256", "refinement", "spaces", "rate_provenance"):
    print(f"{key}: {study[key]}")
for field, errors in study["errors"].items():
    print(field, "errors:", errors)
    print(field, "successive orders:", study["rates"][field])

## 2. Plot physical errors and observed orders

The left panel shows measured errors; the right shows successive rates. Dotted lines display only independently stated target orders. Sparse labelled sample sizes and unlabelled log minor ticks remain legible at documentation width. Vector SVG/PDF exports retain scalable text and curves.

In [ ]:
figure = plot_method_series(study)
plt.show()
output = ROOT / "build" / "method-convergence"
output.mkdir(parents=True, exist_ok=True)
for extension in ("png", "svg", "pdf"):
    figure.savefig(output / f"{method}-convergence.{extension}", dpi=140)

## 3. Compare all qualified final intervals

Inspect more than one final interval. Discarding an inconvenient physical observable or changing its norm can obscure an unresolved approximation error. A fine local discretization must be controlled independently of skeletal refinement.

In [ ]:
for study in studies:
    print("\n", study["method"], "—", study["refinement"])
    for field, rates in study["rates"].items():
        target = study["expected"].get(field, "reported observation")
        print(field, "target:", target, "last intervals:", rates[-3:])

## References and scope

Each method's complete original references appear on its [rendered tutorial](https://ipes-lncc.github.io/pymhm/tutorials/methods/) and in the [Bibliography](https://ipes-lncc.github.io/pymhm/literature/). The raw records retain their execution identities. Smooth energy orders do not apply automatically to SPE10 or a coefficient interface. Unfitted trace refinement holds the macro mesh fixed, and time refinement holds the spatial operator fixed. Reconstruction and adaptive tutorials verify conservation, effectivity or error decrease instead of inventing an unrelated uniform-mesh order.